# Stage 2b — three protein encoders, one controlled comparison

Caches protein embeddings from three encoders so the ablation can be re-run on each
with everything else held fixed.

| pass | encoder | why |
|---|---|---|
| 1 | `facebook/esm2_t33_650M_UR50D` | the real candidate — 650M params, trained on UniRef50 |
| 2 | `facebook/esm2_t30_150M_UR50D` | free scaling point. If 150M beats ProtBERT's 420M, the story is data + architecture, not size |
| 3 | `Rostlab/prot_bert` @ 1024 | **the control.** Same length cap as ESM-2 |

**Why pass 3 exists.** Our original ProtBERT run used `max_length=3200`; ESM-2 was
trained at 1024. Comparing them directly would confound "better encoder" with "sees more
of the protein". Re-running ProtBERT at 1024 makes the encoder the only thing that
differs. Without it we would have an argument rather than a result.

Ligands are skipped in all three passes — swapping the protein encoder does not change
them, and the existing `mol_*.npy` cache is reused.

Runtime: ~30 min for 650M, ~8 for 150M, ~10 for ProtBERT. Output ~200 MB total.

**Runtime → Change runtime type → L4 GPU** before you start.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers>=4.44,<6"
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda:", torch.cuda.is_available())

## Upload

Drag in three files: `scripts/02_embed.py`, `data/processed/proteins.parquet`,
`data/processed/ligands.parquet`. (`ligands.parquet` is only read for a row count,
but the script expects it to exist.)

In [ ]:
from google.colab import files
up = files.upload()
for n in up: print(" uploaded:", n)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/plapt-experiment/emb'
import os; os.makedirs(OUT, exist_ok=True)
print("writing to:", OUT)

## Pass 1 — ESM-2 650M

`--exclude-special` drops `[CLS]`/`[SEP]` and `<cls>`/`<eos>` from the mean pool, so the
average is over real residues only. Encoders add different special tokens, so leaving
them in would make a cross-encoder comparison partly about tokenizer bookkeeping.

Watch the `NOTE:` line at the end — it reports how many proteins exceeded the length cap
and were truncated. Expect ~7% at 1024.

In [ ]:
!python 02_embed.py --data . --out "$OUT" --skip-ligands --exclude-special     --prot-model facebook/esm2_t33_650M_UR50D --prot-max 1024 --tag _esm650     --prot-budget 8192

## Pass 2 — ESM-2 150M

In [ ]:
!python 02_embed.py --data . --out "$OUT" --skip-ligands --exclude-special     --prot-model facebook/esm2_t30_150M_UR50D --prot-max 1024 --tag _esm150     --prot-budget 16384

## Pass 3 — ProtBERT at 1024, the control

Same cap, same special-token handling, same everything. Only the encoder differs from
passes 1 and 2.

In [ ]:
!python 02_embed.py --data . --out "$OUT" --skip-ligands --exclude-special     --prot-model Rostlab/prot_bert --prot-max 1024 --tag _pb1024     --prot-budget 16384

## Compare the representations before you leave

Effective rank is how many independent directions each representation actually uses. The
original ProtBERT `pooler_output` scored **1.00 / 1024** — every protein on one line.
This is the cheap check on whether a new encoder has the same problem.

In [ ]:
import numpy as np, os
rows = []
for tag, dim in [('_esm650',1280), ('_esm150',640), ('_pb1024',1024)]:
    for kind in ['pooler','mean']:
        f = os.path.join(OUT, f'prot_{kind}{tag}.npy')
        if not os.path.exists(f): continue
        a = np.load(f).astype('float64')
        if not np.abs(a).sum():
            rows.append((f'{kind}{tag}', dim, float('nan'), float('nan'))); continue
        if len(a) > 4000:
            a = a[np.random.default_rng(0).choice(len(a), 4000, replace=False)]
        s = np.linalg.svd(a - a.mean(0), compute_uv=False)
        p = s**2/(s**2).sum()
        eff = float(np.exp(-(p*np.log(p+1e-300)).sum()))
        N = a/np.maximum(np.linalg.norm(a,axis=1,keepdims=True),1e-12)
        sub = N[:1500]; C = sub@sub.T
        cos = C[np.triu_indices(len(sub),1)].mean()
        rows.append((f'{kind}{tag}', a.shape[1], eff, cos))

print(f"{'representation':22s} {'dims':>5s} {'eff.rank':>9s} {'mean cos':>9s}")
for n,d,e,c in rows:
    print(f"{n:22s} {d:>5d} {e:>9.2f} {c:>9.4f}")
print()
print("for reference, the original ProtBERT@3200 run:")
print(f"{'pooler (PLAPT)':22s} {1024:>5d} {1.00:>9.2f} {1.0000:>9.4f}")
print(f"{'mean':22s} {1024:>5d} {12.71:>9.2f} {0.7808:>9.4f}")

In [ ]:
import os
for f in sorted(os.listdir(OUT)):
    print(f"{os.path.getsize(os.path.join(OUT,f))/1e6:9.1f} MB  {f}")

## Then, locally

Copy `prot_*_esm650.npy`, `prot_*_esm150.npy`, `prot_*_pb1024.npy` and the
`embed_meta_*.json` files into `data/processed/`, then run the ablation once per encoder:

```powershell
foreach ($t in '_esm650','_esm150','_pb1024') {
  .venv\Scripts\python.exe -I scripts_train_ablation.py `
     --pooling mean --prot-tag $t --arms full,mol_zero,prot_zero `
     --seeds 3 --max-epochs 60 --patience 6 --threads 10 --tag $t
}
```

Three arms is enough: `full` and `mol_zero` give the protein's contribution, `prot_zero`
gives the protein-only ceiling. The shuffle and cut variants already agreed with `zero`
in the main grid.

Compare each against the published ProtBERT@3200 numbers: `full` 0.7414, `mol_zero`
0.5507, `prot_zero` 0.5689.